# LAB 03 — Word Embeddings

Notebook này dùng **Brown Corpus** của NLTK. Brown là corpus tiếng Anh có nhiều thể loại.


## 1. Cài đặt thư viện

In [27]:
%pip install nltk gensim pandas numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\Users\ADMIN\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 2. Tải và đọc Brown Corpus

In [28]:
import nltk
from nltk.corpus import brown

try:
    brown.sents()  # Kiểm tra corpus đã có chưa
except LookupError:
    nltk.download('brown')

MAX_SENTENCES = 10_000
sentences = brown.sents()[:MAX_SENTENCES]

# Chuẩn hóa token: chữ thường, chỉ giữ token không rỗng
corpus = [[str(token).lower() for token in sentence if str(token).strip()]
          for sentence in sentences]
corpus = [sentence for sentence in corpus if sentence]

print(f"Số câu sử dụng: {len(corpus):,}")
print(f"Ví dụ câu đầu tiên: {' '.join(corpus[0]) if corpus else 'Không có dữ liệu'}")

Số câu sử dụng: 10,000
Ví dụ câu đầu tiên: the fulton county grand jury said friday an investigation of atlanta's recent primary election produced `` no evidence '' that any irregularities took place .


## 3. Phần 17–18 — Huấn luyện Word2Vec và kiểm tra từ gần nghĩa

In [29]:
from gensim.models import Word2Vec

VECTOR_SIZE = 100
WINDOW = 5
MIN_COUNT = 1
EPOCHS = 5
SG = 0  # 0 = CBOW, 1 = Skip-gram

model = Word2Vec(
    sentences=corpus,
    vector_size=VECTOR_SIZE,
    window=WINDOW,
    min_count=MIN_COUNT,
    workers=4,
    sg=SG,
    seed=42,
    epochs=EPOCHS
)

print('Corpus: NLTK Brown Corpus, tối đa', MAX_SENTENCES, 'câu')
print('Vocabulary size:', len(model.wv))
print('Embedding dimension:', VECTOR_SIZE)
print('Window:', WINDOW)
print('min_count:', MIN_COUNT)
print('Epochs:', EPOCHS)
print('Objective:', 'CBOW' if SG == 0 else 'Skip-gram')

Corpus: NLTK Brown Corpus, tối đa 10000 câu
Vocabulary size: 21248
Embedding dimension: 100
Window: 5
min_count: 1
Epochs: 5
Objective: CBOW


In [30]:
# Kiểm tra từ doctor và các từ gần nhất nếu doctor có trong vocabulary
if 'doctor' in model.wv:
    print(model.wv.most_similar('doctor', topn=10))
else:
    print("Từ 'doctor' không có trong vocabulary của mô hình này.")

[('street', 0.9911569952964783), ('price', 0.9911369681358337), ('miss', 0.9909815788269043), ('sen.', 0.9908357858657837), ('john', 0.9907927513122559), ('25', 0.9907463192939758), ('nov.', 0.9907411932945251), ('worth', 0.9907162189483643), ('martin', 0.9907147884368896), ('&', 0.9907124638557434)]


## 4. Phần 19 — Thử các context window 2, 5, 10

In [31]:
window_models = {}
for window_size in [2, 5, 10]:
    window_models[window_size] = Word2Vec(
        sentences=corpus,
        vector_size=100,
        window=window_size,
        min_count=1,
        workers=4,
        sg=0,
        seed=42,
        epochs=5
    )
    print(f"window={window_size}: vocab={len(window_models[window_size].wv)}")

window=2: vocab=21248
window=5: vocab=21248
window=10: vocab=21248


## 5. Phần 20 — Thử embedding dimensions 50, 100, 300

In [32]:
dimension_models = {}
for dimension in [50, 100, 300]:
    dimension_models[dimension] = Word2Vec(
        sentences=corpus,
        vector_size=dimension,
        window=5,
        min_count=1,
        workers=4,
        sg=0,
        seed=42,
        epochs=5
    )
    print(f"dimension={dimension}: vocab={len(dimension_models[dimension].wv)}")

dimension=50: vocab=21248
dimension=100: vocab=21248
dimension=300: vocab=21248


## 6. Phần 21 — Word similarity pairs

Tính cosine similarity cho các cặp được yêu cầu. Nếu một từ không có trong vocabulary thì ghi `None` thay vì tạo ra điểm giả. Vì corpus nhỏ và có thể thiếu một số từ hiếm, một vài cặp có thể không tính được hoặc cho điểm chưa ổn định.

In [33]:
import pandas as pd

word_pairs = [
    ('doctor', 'physician'),
    ('car', 'automobile'),
    ('king', 'queen'),
    ('cat', 'dog'),
    ('computer', 'banana'),
    ('doctor', 'nurse'),
    ('doctor', 'hospital'),
    ('doctor', 'disease'),
]

similarity_rows = []
for word1, word2 in word_pairs:
    if word1 in model.wv and word2 in model.wv:
        score = float(model.wv.similarity(word1, word2))
        status = 'computed'
    else:
        score = None
        missing = [word for word in (word1, word2) if word not in model.wv]
        status = 'missing: ' + ', '.join(missing)
    similarity_rows.append({
        'word1': word1,
        'word2': word2,
        'cosine_similarity': score,
        'status': status
    })

similarity_df = pd.DataFrame(similarity_rows)
similarity_df

,word1,word2,cosine_similarity,status
0,doctor,physician,0.974778,computed
1,car,automobile,0.991938,computed
2,king,queen,0.994707,computed
3,cat,dog,0.714646,computed
4,computer,banana,NaN,"missing: computer, banana"
5,doctor,nurse,0.962981,computed
6,doctor,hospital,0.987286,computed
7,doctor,disease,0.982228,computed


In [39]:
# Lấy một số câu từ corpus để làm evidence
target_words = [
    'doctor',
    'physician',
    'nurse',
    'hospital',
    'disease',
    'car',
    'automobile',
    'king',
    'queen',
    'cat',
    'dog'
]

for target in target_words:
    print(f"\n===== {target} =====")
    count = 0

    for sentence in corpus:
        if target in sentence:
            print(' '.join(sentence))
            count += 1

            if count >= 5:
                break


===== doctor =====
four additional deputies be employed at the fulton county jail and `` a doctor , medical intern or extern be employed for night and weekend duty at the jail '' .
dr. clark holds an earned doctor of education degree from the university of oklahoma .
every person will choose his own doctor and hospital '' .
the plan does not cover doctor bills .
if the doctor is conscientious , he wants to study the patient .

===== physician =====
while working out in sylvania a swelling developed in the knee and he came here to consult the team physician .
mr. pezza was taken to a nearby johnston physician , dr. allan a. disimone , who treated him .
first of all , the admitting physician in the va hospital gets the patient as a new patient .
as a result , it takes a little longer than it would on the outside where the family physician knows about the patient .
secondly , the va physician knows that when the patient leaves the hospital , he is no longer going to have a chance to visi

In [40]:
# Lấy thêm evidence cho các trường hợp bất thường
targets = [
    'doctor',
    'representative',
    'father',
    'grants',
    '1959',
    'reply'
]

for target in targets:
    print(f"\n===== {target} =====")
    count = 0

    for sentence in corpus:
        if target in sentence:
            print(' '.join(sentence))
            count += 1

            if count >= 5:
                break


===== doctor =====
four additional deputies be employed at the fulton county jail and `` a doctor , medical intern or extern be employed for night and weekend duty at the jail '' .
dr. clark holds an earned doctor of education degree from the university of oklahoma .
every person will choose his own doctor and hospital '' .
the plan does not cover doctor bills .
if the doctor is conscientious , he wants to study the patient .

===== representative =====
calling the democrats the `` party that lives , breathes and thinks for the good of the people '' , hughes asked , `` a representative democratic vote in the primary for a springboard toward victory in november '' .
in an apparent effort to head off such a rival primary slate , mr. wagner talked by telephone yesterday with representative charles a. buckley , the bronx democratic leader , and with joseph t. sharkey , the brooklyn democratic leader .
the panel's action depends on the return of representative james w. trimble , democrat o

## 7. Phần 22 — Word analogy (tùy chọn)

Thử các phép analogy nếu tất cả từ đầu vào đều có trong vocabulary. Kết quả chỉ là dự đoán của mô hình, không phải đáp án được đảm bảo đúng.

In [41]:
analogy_tests = [
    ('king - man + woman', ['king', 'woman'], ['man']),
    ('Paris - France + Italy', ['Paris', 'Italy'], ['France']),
]

for label, positive, negative in analogy_tests:
    needed = positive + negative
    missing = [word for word in needed if word not in model.wv]
    print(f"\n{label}")
    if missing:
        print('Không thể tính; thiếu từ:', missing)
    else:
        print(model.wv.most_similar(positive=positive, negative=negative, topn=5))


king - man + woman
[('23', 0.9944818615913391), ('a.', 0.9942857623100281), ('sept.', 0.9942758679389954), ('.', 0.9942740201950073), ('c.', 0.994203507900238)]

Paris - France + Italy
Không thể tính; thiếu từ: ['Paris', 'Italy', 'France']


## 8. Lưu kết quả similarity vào CSV

In [42]:
similarity_df.to_csv('results.csv', index=False)
print('Đã lưu results.csv')

Đã lưu results.csv
